In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
#rms prop
class RMSProp:
    def __init__(self, params, lr = 1e-4, beta = 0.999, eps=1e-8):
        self.params = list(params)
        self.alpha = lr
        self.beta = beta
        self.eps = eps
        self.velocity = [
            torch.zeros_like(p)
            for p in self.params
        ]

    @torch.no_grad()
    def step(self):
        velocity = self.velocity
        b = self.beta
        lr = self.alpha
        params = self.params

        for p, v in zip(params, velocity):
            if p.grad is not None:
                v[:] = b*v + (1-b)*(p.grad**2) # update in place via slicing [:]

                p[:] -= lr*p.grad/( v**0.5 + self.eps)

    def zero_grad(self, set_to_none=True):
        for p in self.params:
            if p.grad is None:
                continue
                
            if set_to_none:
                p.grad = None 
            else:
                p.grad.zero()
                
        

In [3]:
#adam w 
class AdamW:
    def __init__(
        self, 
        params, 
        lr = 1e-4, 
        beta1 = 0.9, 
        beta2 = 0.999, 
        eps=1e-8, 
        decay = 0.1
        ):
        
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.decay = decay
        
        self.momentum = [
        torch.zeros_like(p)
        for p in self.params
        ]
        
        self.velocity = [
        torch.zeros_like(p)
        for p in self.params
        ]

    @torch.no_grad()
    def step(self):
        momentum = self.momentum
        velocity = self.velocity
        b1 = self.beta1
        b2 = self.beta2
        lr = self.lr
        params = self.params
        self.t += 1

        for p, m, v in zip(params, momentum, velocity):
            if p.grad is not None:

                # m[:] = b1*m + (1-b1)*(p.grad)    # update in place via slicing [:]
                # v[:] = b2*v + (1-b2)*(p.grad**2) # update in place via slicing [:]
                m.mul_(b1).add_( (1-b1) * p.grad )
                v.mul_(b2).add_( (1-b2) * p.grad.square() )
                

                #bias correction
                m_hat = m/(1 - b1**self.t)
                v_hat = v/(1 - b2**self.t)
                
                # p[:] -= lr*m_hat/( v_hat**0.5 + self.eps
                p.sub_( lr*m_hat/( v_hat**0.5 + self.eps) )

                # AdamW weight decay
                p.mul_(1-self.decay*lr)
        

    def zero_grad(self, set_to_none=True):
        for p in self.params:
            if p.grad is None:
                continue
                
            if set_to_none:
                p.grad = None 
            else:
                p.grad.zero_()
                

In [4]:
#lion
class Lion:
    def __init__(
        self, 
        params, 
        lr = 1e-4, 
        beta1 = 0.9, 
        beta2 = 0.99, 
        decay = 0.1
        ):
        
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.decay = decay
        
        self.momentum = [
        torch.zeros_like(p)
        for p in self.params
        ]
        

    @torch.no_grad()
    def step(self):
        momentum = self.momentum
        b1 = self.beta1
        b2 = self.beta2
        lr = self.lr
        params = self.params

        for p, m in zip(params, momentum):
            if p.grad is not None:
                #blend current gradient with stored momentum via beta
                update = b1*m + (1-b1)*p.grad
                
                #decoupled weight decay
                p.mul_(1- self.decay*lr)

                #update each weight by only the directions -1, 0, or +1
                p.add_( update.sign(), alpha= -lr)

                #update the stored momentum for the next step
                m.mul_(b2).add_( (1-b2)*p.grad )

    def zero_grad(self, set_to_none=True):
        for p in self.params:
            if p.grad is None:
                continue
                
            if set_to_none:
                p.grad = None 
            else:
                p.grad.zero_()
                